## 1. Import Libraries




In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sqlalchemy import create_engine
import glob
import os
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import statsmodels.api as sm

## 2. Load and Prepare Dataset

In [2]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")

    df = pd.read_sql_query("SELECT * FROM all_regions", engine)
    combined_df = df.copy()
    print(f"Loaded {len(df):,} records from all_regions.")
except Exception as error:
    raise RuntimeError(
        "Could not connect to MySQL or load the all_regions table. "
        "Check that MySQL is running and the air_quality database exists."
    ) from error

Connected to air_quality!
Loaded 418,753 records from all_regions.


## 3. Descriptive Statistics

Calculate mean, median, standard deviation, variance, min, max, Q1, Q3, and IQR for each pollutant. Describe the overall numerical characteristics of the air-quality data.

In [5]:
pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

descriptive_stats = df[pollutants].agg([
    "count",
    "mean",
    "median",
    "std",
    "var",
    "min",
    "max"
])

q1 = df[pollutants].quantile(0.25)
q3 = df[pollutants].quantile(0.75)

descriptive_stats.loc["Q1"] = q1
descriptive_stats.loc["Q3"] = q3
descriptive_stats.loc["IQR"] = q3 - q1

descriptive_stats = descriptive_stats.T

descriptive_stats

,count,mean,median,std,var,min,max,Q1,Q3,IQR
PM2.5,417394.0,79.842636,55.0,81.019792,6.564207e+03,2.0000,999.0,20.0000,111.0,91.0000
PM10,418454.0,104.835515,82.0,92.443437,8.545789e+03,2.0000,999.0,36.0000,145.0,109.0000
SO2,416852.0,15.868837,7.0,21.756296,4.733364e+02,0.2856,500.0,3.0000,20.0,17.0000
NO2,415220.0,50.536439,43.0,35.183673,1.237891e+03,1.0265,290.0,23.0000,71.0,48.0000
CO,408757.0,1233.904314,900.0,1170.754170,1.370665e+06,100.0000,10000.0,500.0000,1500.0,1000.0000
O3,415317.0,57.116693,45.0,56.651137,3.209351e+03,0.2142,1071.0,10.0674,82.0,71.9326


In [6]:
confidence_results = []

for pollutant in pollutants:
    values = df[pollutant].dropna()
    n = len(values)
    mean = values.mean()
    standard_error = values.std() / np.sqrt(n)
    ci_low, ci_high = sm.stats.DescrStatsW(values).tconfint_mean(
        alpha=0.05
    )

    confidence_results.append({
        "Pollutant": pollutant,
        "Mean": mean,
        "95% CI Lower": ci_low,
        "95% CI Upper": ci_high
    })

confidence_df = pd.DataFrame(confidence_results)

confidence_df

,Pollutant,Mean,95% CI Lower,95% CI Upper
0,PM2.5,79.842636,79.596844,80.088428
1,PM10,104.835515,104.555422,105.115607
2,SO2,15.868837,15.802791,15.934882
3,NO2,50.536439,50.429422,50.643455
4,CO,1233.904314,1230.315239,1237.493388
5,O3,57.116693,56.944400,57.288987


## 4. Station-Level Analysis 


In [7]:
station_stats = df.groupby("station")[pollutants].agg([
    "mean",
    "median",
    "std"
])

station_stats

PM2.5                              PM10                    \
                    mean     median        std        mean median        std   
station                                                                        
Aotizhongxin   82.813909  58.000000  82.271178  110.188808   87.0  95.664991   
Changping      71.216180  46.702381  72.611162   94.915441   72.0  84.093508   
Dingling       66.185015  41.000000  72.691094   84.020313   60.0  80.221875   
Dongsi         86.113104  61.000000  86.560947  110.427394   86.0  98.488701   
Guanyuan       82.975671  59.000000  81.161300  109.216305   89.0  92.230893   
Gucheng        83.922580  60.000000  82.935287  119.137933  100.0  97.529837   
Huairou        69.554026  46.000000  71.318680   91.758388   69.0  84.382682   
Nongzhanguan   85.115528  59.000000  86.687041  109.461446   86.0  96.057586   
Shunyi         79.659489  55.000000  81.619813   99.081751   77.0  90.243928   
Tiantan        82.003230  58.000000  80.934024  106.539151   85.0  90.327850   
Wanliu         83.428047  59.000000  82.087592  110.676376   88.0  93.464734   
Wanshouxigong  84.976929  60.000000  86.071200  112.426760   91.0  98.185428   

                     SO2                          NO2                        \
                    mean median        std       mean     median        std   
station                                                                       
Aotizhongxin   17.391519    9.0  22.847207  59.302897  53.624983  37.117759   
Changping      14.989729    7.0  21.059629  44.182105  36.000000  29.547959   
Dingling       11.784493    5.0  15.604248  27.380357  19.000000  26.345170   
Dongsi         18.550229   10.0  22.997806  53.620078  47.000000  33.936942   
Guanyuan       17.633102    8.0  23.670394  57.974535  51.000000  35.230000   
Gucheng        15.452883    7.0  21.699728  55.867292  50.000000  36.530782   
Huairou        12.142057    4.0  18.976046  32.121847  25.000000  26.395330   
Nongzhanguan   18.781994    9.0  24.397342  58.154098  51.000000  36.409474   
Shunyi         13.553864    5.0  19.557829  43.831635  37.000000  30.928364   
Tiantan        14.436874    7.0  20.216801  53.163019  47.000000  32.007531   
Wanliu         18.420137   10.0  22.684980  65.340552  60.000000  38.112252   
Wanshouxigong  17.219534    8.0  24.068333  55.493035  48.000000  35.888420   

                        CO                              O3                      
                      mean  median          std       mean   median        std  
station                                                                         
Aotizhongxin   1268.263377   900.0  1246.287996  55.788890  42.0000  57.653289  
Changping      1159.020549   800.0  1122.527930  57.892060  46.0000  54.305769  
Dingling        906.963471   600.0   903.516539  68.640764  61.0000  54.222805  
Dongsi         1332.986414  1000.0  1198.638817  57.308912  44.4465  58.074952  
Guanyuan       1268.181750   900.0  1165.006728  55.280380  40.0000  57.308633  
Gucheng        1327.042281   900.0  1215.983387  57.670740  44.7678  57.005423  
Huairou        1021.557389   800.0   896.592292  59.738890  49.0000  54.580149  
Nongzhanguan   1332.638499   900.0  1264.546064  58.443926  44.5536  58.392400  
Shunyi         1190.779680   800.0  1171.915986  54.727617  43.0000  54.804220  
Tiantan        1301.408650   900.0  1179.751867  55.965753  40.0000  59.160000  
Wanliu         1326.363803   900.0  1281.903424  47.872996  30.0000  54.733887  
Wanshouxigong  1372.190581  1000.0  1230.772371  55.962645  42.0000  57.082378

In [8]:
station_summary = []

for station, group in df.groupby("station"):
    for pollutant in pollutants:
        values = group[pollutant].dropna()
        station_summary.append({
            "Station": station,
            "Pollutant": pollutant,
            "Mean": values.mean(),
            "Median": values.median(),
            "Standard Deviation": values.std()
        })

station_summary_df = pd.DataFrame(station_summary)

station_summary_df

,Station,Pollutant,Mean,Median,Standard Deviation
0,Aotizhongxin,PM2.5,82.813909,58.000000,82.271178
1,Aotizhongxin,PM10,110.188808,87.000000,95.664991
2,Aotizhongxin,SO2,17.391519,9.000000,22.847207
3,Aotizhongxin,NO2,59.302897,53.624983,37.117759
4,Aotizhongxin,CO,1268.263377,900.000000,1246.287996
...,...,...,...,...,...
67,Wanshouxigong,PM10,112.426760,91.000000,98.185428
68,Wanshouxigong,SO2,17.219534,8.000000,24.068333
69,Wanshouxigong,NO2,55.493035,48.000000,35.888420
70,Wanshouxigong,CO,1372.190581,1000.000000,1230.772371


In [9]:
highest_mean_results = []

for pollutant in pollutants:
    station_means = df.groupby("station")[pollutant].mean()
    highest_station = station_means.idxmax()
    highest_mean = station_means.max()

    highest_mean_results.append({
        "Pollutant": pollutant,
        "Station with Highest Mean": highest_station,
        "Mean Concentration": highest_mean
    })

highest_mean_df = pd.DataFrame(highest_mean_results)

highest_mean_df

,Pollutant,Station with Highest Mean,Mean Concentration
0,PM2.5,Dongsi,86.113104
1,PM10,Gucheng,119.137933
2,SO2,Nongzhanguan,18.781994
3,NO2,Wanliu,65.340552
4,CO,Wanshouxigong,1372.190581
5,O3,Dingling,68.640764


In [10]:
station_confidence = []

for station, group in df.groupby("station"):
    for pollutant in pollutants:
        values = group[pollutant].dropna()

        if len(values) > 1:
            ci_low, ci_high = sm.stats.DescrStatsW(
                values
            ).tconfint_mean(alpha=0.05)

            station_confidence.append({
                "Station": station,
                "Pollutant": pollutant,
                "Mean": values.mean(),
                "95% CI Lower": ci_low,
                "95% CI Upper": ci_high
            })

station_confidence_df = pd.DataFrame(station_confidence)

pd.set_option("display.max_rows", None)

## 5. ANOVA Explanation

One-way ANOVA tests whether the mean concentration of each pollutant differs among the 12 monitoring stations. For each pollutant, the null hypothesis states that all station means are equal, while the alternative hypothesis states that at least one station mean differs. The analysis uses the `all_regions` table from the MySQL `air_quality` database and applies a Bonferroni correction across the six pollutant tests.

In [14]:
from scipy.stats import f_oneway, levene, normaltest
from statsmodels.stats.multicomp import pairwise_tukeyhsd

pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

for pollutant in pollutants:
    df[pollutant] = pd.to_numeric(df[pollutant], errors="coerce")

anova_results = []

for pollutant in pollutants:
    station_groups = [
        group[pollutant].dropna().to_numpy()
        for _, group in df.groupby("station", sort=True)
    ]
    f_statistic, p_value = f_oneway(*station_groups)
    anova_results.append({
        "pollutant": pollutant,
        "stations": len(station_groups),
        "observations": sum(len(values) for values in station_groups),
        "F-statistic": f_statistic,
        "p-value": p_value,
    })

anova_results = pd.DataFrame(anova_results)
anova_results["p-value (Bonferroni)"] = multipletests(
    anova_results["p-value"], method="bonferroni"
)[1]
anova_results["significant (alpha=0.05)"] = (
    anova_results["p-value (Bonferroni)"] < 0.05
)

anova_results.round({
    "F-statistic": 3,
    "p-value": 6,
    "p-value (Bonferroni)": 6,
})

,pollutant,stations,observations,F-statistic,p-value,p-value (Bonferroni),significant (alpha=0.05)
0,PM2.5,12,417394,248.412,0.0,0.0,True
1,PM10,12,418454,425.573,0.0,0.0,True
2,SO2,12,416852,461.828,0.0,0.0,True
3,NO2,12,415220,4048.808,0.0,0.0,True
4,CO,12,408757,507.529,0.0,0.0,True
5,O3,12,415317,236.629,0.0,0.0,True


## 6. ANOVA Interpretation

The one-way ANOVA results show statistically significant differences in mean concentration among the 12 monitoring stations for **all six pollutants**: PM2.5, PM10, SO2, NO2, CO, and O3. Every Bonferroni-adjusted p-value is below 0.05, so the null hypothesis that all station means are equal is rejected for every pollutant.

The strongest station-level evidence is for **NO2 (F = 4048.808)**. The remaining F-statistics range from **236.629 for O3** to **507.529 for CO**, confirming that station mean concentrations vary substantially across pollutants. These results show an association between monitoring station and average pollutant concentration, but they do not establish causation or identify which station pairs differ.

The following post-hoc section uses Tukey's HSD to identify the specific station pairs responsible for these overall ANOVA results.

## 7. Post-Hoc Explanation

A significant ANOVA result shows that at least one station mean differs, but it does not identify the station pairs responsible for the differences. Tukey's HSD compares every pair of stations for each pollutant while controlling the family-wise error rate at 0.05.

The same analysis checks two ANOVA assumptions: Levene's test evaluates whether station variances are similar, and the residual normality test evaluates whether residuals are approximately normal. The full pairwise results are retained in `significant_pairwise_results`, while the next cell displays only a compact summary.

## 8. Compact Post-Hoc Results

The output shows the number of significant station pairs for each pollutant, the three largest significant mean differences per pollutant, and the assumption-test results.

In [15]:
significant_pairwise_results = []
assumption_results = []

for pollutant in pollutants:
    valid_data = df[["station", pollutant]].dropna()

    tukey = pairwise_tukeyhsd(
        endog=valid_data[pollutant],
        groups=valid_data["station"],
        alpha=0.05,
    )
    tukey_table = pd.DataFrame(
        tukey.summary().data[1:],
        columns=tukey.summary().data[0],
    )
    tukey_table = tukey_table[tukey_table["reject"] == True].copy()
    tukey_table.insert(0, "pollutant", pollutant)
    significant_pairwise_results.append(tukey_table)

    station_values = [
        group[pollutant].dropna().to_numpy()
        for _, group in df.groupby("station", sort=True)
    ]
    levene_statistic, levene_p_value = levene(*station_values, center="median")

    residuals = valid_data[pollutant] - valid_data.groupby("station")[pollutant].transform("mean")
    residual_sample = residuals.sample(
        n=min(5000, len(residuals)), random_state=42
    )
    normality_statistic, normality_p_value = normaltest(residual_sample)

    assumption_results.append({
        "pollutant": pollutant,
        "Levene statistic": levene_statistic,
        "Levene p-value": levene_p_value,
        "Residual normality statistic": normality_statistic,
        "Residual normality p-value": normality_p_value,
        "equal variances at alpha=0.05": levene_p_value >= 0.05,
        "normal residuals at alpha=0.05": normality_p_value >= 0.05,
    })

significant_pairwise_results = pd.concat(
    significant_pairwise_results, ignore_index=True
)
assumption_results = pd.DataFrame(assumption_results)

pairwise_counts = (
    significant_pairwise_results.groupby("pollutant")
    .size()
    .reset_index(name="significant pair count")
)

top_pairwise_differences = (
    significant_pairwise_results.assign(
        absolute_meandiff=lambda table: table["meandiff"].abs()
    )
    .sort_values(["pollutant", "absolute_meandiff"], ascending=[True, False])
    .groupby("pollutant", group_keys=False)
    .head(3)
    [["pollutant", "group1", "group2", "meandiff", "p-adj", "lower", "upper"]]
)

print("Number of significant Tukey HSD pairs by pollutant")
display(pairwise_counts)

print("Three largest significant mean differences per pollutant")
display(top_pairwise_differences)

print("Assumption test results")
display(assumption_results.round(6))

Number of significant Tukey HSD pairs by pollutant


,pollutant,significant pair count
0,CO,57
1,NO2,62
2,O3,47
3,PM10,53
4,PM2.5,48
5,SO2,58


Three largest significant mean differences per pollutant


,pollutant,group1,group2,meandiff,p-adj,lower,upper
249,CO,Dingling,Wanshouxigong,465.2271,0.0,436.1152,494.3391
241,CO,Dingling,Dongsi,426.0229,0.0,396.4575,455.5884
245,CO,Dingling,Nongzhanguan,425.6750,0.0,396.5668,454.7832
186,NO2,Dingling,Wanliu,37.9602,0.0,37.1289,38.7915
209,NO2,Huairou,Wanliu,33.2187,0.0,32.3863,34.0511
160,NO2,Aotizhongxin,Dingling,-31.9225,0.0,-32.7532,-31.0918
299,O3,Dingling,Wanliu,-20.7678,0.0,-22.1760,-19.3595
297,O3,Dingling,Shunyi,-13.9131,0.0,-15.3197,-12.5066
293,O3,Dingling,Guanyuan,-13.3604,0.0,-14.7652,-11.9556
66,PM10,Dingling,Gucheng,35.1176,0.0,32.8437,37.3915


Assumption test results


,pollutant,Levene statistic,Levene p-value,Residual normality statistic,Residual normality p-value,equal variances at alpha=0.05,normal residuals at alpha=0.05
0,PM2.5,120.896315,0.0,2065.633031,0.0,False,False
1,PM10,177.608144,0.0,2210.610577,0.0,False,False
2,SO2,298.391913,0.0,3614.154786,0.0,False,False
3,NO2,1051.849259,0.0,687.417394,0.0,False,False
4,CO,210.706003,0.0,2756.980797,0.0,False,False
5,O3,65.483674,0.0,1313.966058,0.0,False,False


## 9. Post-Hoc Interpretation

Tukey's HSD identified **325 significant station-pair differences** across the six pollutants. The compact summary reports the number of significant pairs for each pollutant and the three largest absolute mean differences, while the complete results remain available in `significant_pairwise_results` if detailed inspection is needed.

The largest displayed differences are generally associated with Dingling compared with higher-concentration stations, especially for CO, NO2, PM10, and O3. The `meandiff` value is the difference between the second and first station listed in each pair; a positive value indicates a higher mean for the second station.

Levene's test rejected equal variances and the residual normality test rejected normality for every pollutant. Because the dataset contains hundreds of thousands of hourly observations, these tests are sensitive to small departures from ideal assumptions. The results should therefore be considered together with effect sizes and station means. Temporal dependence among hourly observations also limits the independence assumption, so the findings demonstrate station-level differences rather than causation. Welch's ANOVA and Games-Howell comparisons would be suitable robust follow-up methods.